# Civet Behaviour Detection — Frame Extraction + Pre-processing
**Chiang Mai Zoo · CCTV civet project · Part 1 & 2 of the pipeline**

```
Videos → Annotations → [ FRAME EXTRACTION ] → [ PRE-PROCESSING ] → Labelled frame sequences → Model training → Evaluation
                         ^^^^^^^^^^^^^^^^ this notebook ^^^^^^^^^^^^^^^^
```

**What you hand to the model team at the end**

| Output | What it is |
|---|---|
| `frames_raw/` | Frames pulled from every video at a fixed rate, original resolution. Never edit these. |
| `processed/full640/<label>/` | Cleaned full frames, 640×640 letterbox (for YOLO-style models) |
| `processed/crop224/<label>/` | Cleaned crops centred on the civet, 224×224 (for CNN / CNN+LSTM classifiers) |
| `metadata/manifest.csv` | One row per frame: video, camera, label, time, quality flags, civet box, **train/val/test split** |

**Three rules that matter more than any code here**
1. **Never overwrite raw data.** Each step reads one folder and writes a new one, so any step can be re-run.
2. **Split by video, never by frame.** Frames from the same video are near-copies; a random frame split gives fake 99 % accuracy.
3. **The same pre-processing must run at inference time.** Whatever you do to training frames, the deployed system must do to live CCTV frames.

Run the cells top to bottom. Use a **GPU runtime** (Runtime → Change runtime type → T4 GPU) for the MegaDetector step.

## 0. Put the videos on Google Drive like this

```
MyDrive/CMZ/
└── raw_videos/
    ├── Eating/       E01.mp4  E02.mp4 ...
    ├── Locomotion/   L01.mp4  L02.mp4 ...
    └── Resting/      R01.mp4  R02.mp4 ...
```

* The **folder name is the label**. Spell the class folders exactly the same way every time.
* The **file name becomes the `video_id`**, so keep names short and unique (E01, L01 …). No two videos may share a name, even in different folders.
* Use the **original exported video files** from the CCTV system, not screen recordings or screenshots. (The 5 sample PNGs in the first notebook are 1076×611, 1072×606, 1077×602 — different sizes, which means they were screenshots. Real exports give every frame the same size and full quality.)

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/CMZ")
except ImportError:  # running on your own computer instead of Colab
    BASE_DIR = Path(os.environ.get("CMZ_BASE_DIR", "CMZ"))

RAW_VIDEO_DIR  = BASE_DIR / "raw_videos"        # input: never modified
CONVERTED_DIR  = BASE_DIR / "converted_videos"  # only used if a video needs re-encoding
FRAMES_RAW_DIR = BASE_DIR / "frames_raw"        # step 3 output
PROCESSED_DIR  = BASE_DIR / "processed"         # step 8 output
REVIEW_DIR     = BASE_DIR / "review"            # frames a human should look at
METADATA_DIR   = BASE_DIR / "metadata"          # all CSV files

for folder in [RAW_VIDEO_DIR, CONVERTED_DIR, FRAMES_RAW_DIR, PROCESSED_DIR, REVIEW_DIR, METADATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)

In [ ]:
# ===================== SETTINGS (change these, not the code below) =====================
SAMPLE_FPS = 5                 # frames kept per second of video (5 = one frame every 0.2 s)
WINDOW_SEC = 1.0               # length of one behaviour window for sequence models (the slide's "1 s classification")
MAX_FRAMES_PER_VIDEO = None    # e.g. 600 so one long video can't dominate a class; None = keep all
JPEG_QUALITY = 95

FULL_SIZE = 640                # letterbox size for full frames
CROP_SIZE = 224                # size of the square crop around the civet
CROP_MARGIN = 0.25             # extra space around the civet box on each side (25 %)
TEXT_MASK_METHOD = "inpaint"   # "inpaint" (your original method) or "fill" (flat grey, ~10x faster)

USE_MEGADETECTOR = True        # find the civet in every frame (needs GPU for speed)
MD_VERSION = "MDV6-yolov10-e"  # good accuracy/speed balance; "MDV6-yolov10-c" is the tiny CPU-friendly one
MD_CONF = 0.2                  # MegaDetector's recommended default; 0.15–0.3 is the usual range

MOTION_THRESHOLD = 0.02        # % of pixels that changed vs previous frame; below this = near-duplicate
N_FOLDS = 5                    # cross-validation folds (by video)
RANDOM_SEED = 42

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".dav", ".h264", ".h265", ".264", ".265"}

## 1. Build the video list (`videos.csv`)

This scans `raw_videos/` and writes `metadata/videos.csv`. You then add two things the code can't know:

* **`camera`** — which room/camera the clip is from (`room1` … `room4`, or names like `feeding_room`). This is needed later to check that the model isn't just recognising the room.
* **`start_sec` / `end_sec`** *(optional)* — trim each clip to the part where the labelled behaviour is actually happening. Leave blank to use the whole video. **Watch every video once.** A "Resting" clip that ends with the civet walking away will teach the model the wrong thing.

Re-running this cell keeps your edits and only adds newly uploaded videos.

In [ ]:
import pandas as pd

videos_csv = METADATA_DIR / "videos.csv"

found = []
for path in sorted(RAW_VIDEO_DIR.rglob("*")):
    if path.is_file() and path.suffix.lower() in VIDEO_EXTS:
        found.append({
            "video_id": path.stem,
            "label": path.parent.name,
            "file": str(path.relative_to(RAW_VIDEO_DIR)),
            "camera": "", "start_sec": "", "end_sec": "", "notes": "",
        })
found_df = pd.DataFrame(found)

if found_df.empty:
    raise FileNotFoundError(f"No videos found in {RAW_VIDEO_DIR}. Check the folder layout in step 0.")
dupes = found_df[found_df.video_id.duplicated(keep=False)]
if len(dupes):
    raise ValueError(f"These file names are used twice — rename them so every video_id is unique:\n{dupes}")

if videos_csv.exists():
    videos_df = pd.read_csv(videos_csv, dtype=str, keep_default_na=False)
    new_rows = found_df[~found_df.video_id.isin(videos_df.video_id)]
    videos_df = pd.concat([videos_df, new_rows], ignore_index=True)
else:
    videos_df = found_df

videos_df.to_csv(videos_csv, index=False)
print("Videos per label:")
print(videos_df.label.value_counts().to_string(), "\n")
videos_df

In [ ]:
# Fill in the camera for each video here (easier than editing the CSV by hand).
# Leave a video out of the dict if you don't know yet.
CAMERA_OF_VIDEO = {
    # "E01": "feeding_room",
    # "L01": "room2",
}
TRIM_OF_VIDEO = {
    # "R03": (5.0, 42.5),   # keep only 5.0 s → 42.5 s of video R03
}

for vid, cam in CAMERA_OF_VIDEO.items():
    videos_df.loc[videos_df.video_id == vid, "camera"] = cam
for vid, (start, end) in TRIM_OF_VIDEO.items():
    videos_df.loc[videos_df.video_id == vid, ["start_sec", "end_sec"]] = [str(start), str(end)]

videos_df.to_csv(videos_csv, index=False)
missing = videos_df[videos_df.camera == ""].video_id.tolist()
print("Videos still missing a camera:", missing if missing else "none")

## 2. Inspect every video before extracting

CCTV exports cause most of the problems in projects like this:

* **Codec** — many NVRs record H.265/HEVC or a proprietary `.dav` format that OpenCV may not read.
* **Frame rate** — CCTV often records at a variable rate (e.g. it drops to 12 fps at night), and the fps stored in the file can be wrong.
* **Resolution** — check all 4 cameras; they may differ.

`ffprobe` (comes with Colab) reads the real facts. If `opencv_can_read` is `False`, the next cell re-encodes that video to standard H.264.

In [ ]:
import cv2
import json
import subprocess

def video_path_for(row):
    # Use the re-encoded copy if one exists, otherwise the original
    converted = CONVERTED_DIR / f"{row['video_id']}.mp4"
    return converted if converted.exists() else RAW_VIDEO_DIR / row["file"]

def probe_video(path):
    # Facts about a video from ffprobe, plus a check that OpenCV can actually read a frame
    cmd = ["ffprobe", "-v", "error", "-select_streams", "v:0",
           "-show_entries", "stream=codec_name,width,height,avg_frame_rate:format=duration",
           "-of", "json", str(path)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        return {"probe_ok": False, "error": result.stderr.strip()[:200]}
    info = json.loads(result.stdout)
    stream = info["streams"][0] if info.get("streams") else {}
    num, den = (stream.get("avg_frame_rate") or "0/1").split("/")
    fps = float(num) / float(den) if float(den) else 0.0

    cap = cv2.VideoCapture(str(path))
    can_read = cap.isOpened() and cap.read()[0]
    cap.release()

    return {
        "probe_ok": True,
        "codec": stream.get("codec_name"),
        "width": stream.get("width"),
        "height": stream.get("height"),
        "fps": round(fps, 2),
        "duration_sec": round(float(info.get("format", {}).get("duration") or 0), 1),
        "opencv_can_read": bool(can_read),
    }

inventory = []
for _, row in videos_df.iterrows():
    facts = probe_video(video_path_for(row))
    inventory.append({"video_id": row.video_id, "label": row.label, "camera": row.camera, **facts})
inventory_df = pd.DataFrame(inventory)
inventory_df["expected_frames"] = (inventory_df.duration_sec * SAMPLE_FPS).round().astype("Int64")
inventory_df.to_csv(METADATA_DIR / "video_inventory.csv", index=False)

print("Total minutes of video per label:")
print((inventory_df.groupby("label").duration_sec.sum() / 60).round(1).to_string(), "\n")
inventory_df

In [ ]:
def convert_to_h264(src, dst):
    # Re-encode any video to H.264 MP4 that OpenCV can read. crf 18 ≈ visually lossless.
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", str(src),
           "-c:v", "libx264", "-crf", "18", "-preset", "veryfast", "-pix_fmt", "yuv420p", "-an", str(dst)]
    subprocess.run(cmd, check=True)

bad = inventory_df[(inventory_df.probe_ok == False) | (inventory_df.opencv_can_read == False)]
for _, inv in bad.iterrows():
    row = videos_df[videos_df.video_id == inv.video_id].iloc[0]
    print("Re-encoding", row.file, "...")
    convert_to_h264(RAW_VIDEO_DIR / row.file, CONVERTED_DIR / f"{row.video_id}.mp4")

print("Nothing to convert." if bad.empty else "Done — re-run the inspection cell above to confirm.")

## 3. Frame extraction

**How many frames?** CCTV records 15–30 frames per second, and neighbouring frames are almost identical. One minute at 25 fps is 1,500 near-copies, which wastes storage and makes a long, boring "Resting" video dominate the dataset. Keeping **5 frames per second** still captures walking and chewing movements, and gives 5 frames per 1-second window for a sequence model (CNN+LSTM, like the pipeline in the slide).

**How the code picks frames.** It reads the video **in order** and keeps a frame whenever 0.2 s of *video time* has passed. Two common beginner mistakes it avoids:
* *"Keep every 5th frame"* breaks when the CCTV frame rate changes or is reported wrongly. Using timestamps doesn't.
* *Jumping with `cap.set(CAP_PROP_POS_FRAMES, n)`* is unreliable on H.264/H.265 video and can return the wrong frame. Reading sequentially is slower but exact.

**Names.** Each frame is saved as `<video_id>_f<frame number>.jpg`, and its exact time is stored in the CSV, so the model team can rebuild ordered sequences. `window_id` groups frames into 1-second windows.

Re-running is safe: videos that were already extracted are skipped.

In [ ]:
from tqdm.auto import tqdm

def to_float_or_none(value):
    return float(value) if str(value).strip() not in ("", "nan", "None") else None

def extract_frames(video_path, video_id, out_dir, sample_fps=SAMPLE_FPS,
                   start_sec=None, end_sec=None, max_frames=None):
    # Save one frame every 1/sample_fps seconds of video time. Returns one record per saved frame.
    out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise IOError(f"OpenCV cannot open {video_path}")
    native_fps = cap.get(cv2.CAP_PROP_FPS) or 0.0

    step = 1.0 / sample_fps
    next_time = start_sec or 0.0
    records = []
    frame_idx = -1

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame_idx += 1

        t = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0          # time of the frame we just read
        if frame_idx > 0 and t <= 0 and native_fps > 0:       # some files have no timestamps
            t = frame_idx / native_fps

        if end_sec is not None and t > end_sec:
            break
        if t + 1e-6 < next_time:
            continue                                          # not time for the next sample yet

        name = f"{video_id}_f{frame_idx:06d}.jpg"
        cv2.imwrite(str(out_dir / name), frame, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
        records.append({"video_id": video_id, "frame_idx": frame_idx,
                        "timestamp_sec": round(t, 3), "raw_file": name,
                        "height": frame.shape[0], "width": frame.shape[1]})

        while next_time <= t + 1e-6:
            next_time += step
        if max_frames and len(records) >= max_frames:
            break

    cap.release()
    return records

In [ ]:
all_frames = []
for _, row in tqdm(list(videos_df.iterrows()), desc="Videos"):
    out_dir = FRAMES_RAW_DIR / row.video_id
    per_video_csv = out_dir / "frames.csv"

    if per_video_csv.exists():                       # already done on a previous run
        records = pd.read_csv(per_video_csv).to_dict("records")
    else:
        records = extract_frames(
            video_path_for(row), row.video_id, out_dir,
            start_sec=to_float_or_none(row.start_sec),
            end_sec=to_float_or_none(row.end_sec),
            max_frames=MAX_FRAMES_PER_VIDEO,
        )
        pd.DataFrame(records).to_csv(per_video_csv, index=False)

    start = to_float_or_none(row.start_sec) or 0.0
    for r in records:
        r.update({
            "label": row.label,
            "camera": row.camera,
            "raw_path": str(Path("frames_raw") / row.video_id / r["raw_file"]),
            "window_id": int((r["timestamp_sec"] - start) // WINDOW_SEC),
        })
    all_frames.extend(records)

frames_df = pd.DataFrame(all_frames)
frames_df.to_csv(METADATA_DIR / "frames_raw.csv", index=False)

print(f"Extracted {len(frames_df):,} frames from {frames_df.video_id.nunique()} videos\n")
print(frames_df.groupby("label").agg(videos=("video_id", "nunique"), frames=("raw_file", "count")).to_string())

## 4. Look at your frames (always do this)

Before any processing, look at a few frames from **every** video. You are checking: is the civet visible? Is the label right? Is it day (colour) or night (infrared, grey)? Where is the timestamp text?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def show_grid(image_paths, titles=None, columns=6, width=18, gray=False):
    rows = max(1, int(np.ceil(len(image_paths) / columns)))
    fig, axes = plt.subplots(rows, columns, figsize=(width, width / columns * 0.7 * rows))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for i, (ax, p) in enumerate(zip(axes, image_paths)):
        img = cv2.imread(str(p))
        if img is None:
            ax.set_title("cannot read", fontsize=8)
            continue
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(titles[i] if titles else Path(p).name, fontsize=8)
    plt.tight_layout()
    plt.show()

for vid, group in frames_df.groupby("video_id"):
    picks = group.iloc[np.linspace(0, len(group) - 1, min(6, len(group))).astype(int)]
    print(f"{vid}  |  label={group.label.iloc[0]}  camera={group.camera.iloc[0] or '?'}  frames={len(group)}")
    show_grid([BASE_DIR / p for p in picks.raw_path], [f"t={t:.1f}s" for t in picks.timestamp_sec])

## 5. Mark the on-screen text for each camera

The timestamp and camera name burned into CCTV frames must be hidden. Otherwise the model can "cheat": it can learn *"clock says 02:00 → Resting"* or *"text says CAM3 → Eating"* instead of looking at the civet.

Your first notebook used one set of boxes for all images. The 4 cameras may place their text differently, so this version allows **one set of boxes per camera**. Boxes are fractions of width/height: `(x1, y1, x2, y2)`.

Run the cell, check that the **red boxes fully cover the text on every camera**, and adjust the numbers if not.

In [ ]:
DEFAULT_TEXT_BOXES = [
    (0.01, 0.04, 0.30, 0.10),   # timestamp, top-left      (from your first notebook)
    (0.72, 0.87, 0.85, 0.93),   # camera name, bottom-right
]
TEXT_BOXES_BY_CAMERA = {
    # "feeding_room": [(0.01, 0.03, 0.32, 0.09), (0.70, 0.88, 0.86, 0.94)],
}

def text_boxes_for(camera):
    return TEXT_BOXES_BY_CAMERA.get(camera, DEFAULT_TEXT_BOXES)

def draw_boxes(frame, boxes):
    out = frame.copy()
    h, w = out.shape[:2]
    for x1, y1, x2, y2 in boxes:
        cv2.rectangle(out, (int(x1 * w), int(y1 * h)), (int(x2 * w), int(y2 * h)), (0, 0, 255), 3)
    return out

one_per_camera = frames_df.groupby(frames_df.camera.replace("", "unknown")).head(1)
fig, axes = plt.subplots(1, len(one_per_camera), figsize=(6 * len(one_per_camera), 4))
for ax, (_, r) in zip(np.atleast_1d(axes), one_per_camera.iterrows()):
    img = cv2.imread(str(BASE_DIR / r.raw_path))
    ax.imshow(cv2.cvtColor(draw_boxes(img, text_boxes_for(r.camera)), cv2.COLOR_BGR2RGB))
    ax.set_title(f"camera: {r.camera or 'unknown'}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Pre-processing functions

These keep your original pipeline (mask text → grayscale → median filter → CLAHE → 3 channels → letterbox) with four fixes:

| Change | Why |
|---|---|
| **Quality is measured on the masked raw frame, before CLAHE** | CLAHE sharpens edges and stretches brightness, so in your first version a blurry or very dark frame still scored as fine. The burned-in text also has sharp edges that fake a high blur score, so measure *after* masking. |
| **Day/night is recorded** (`lighting` = `color_day` / `ir_night`) | Civets are nocturnal, so most activity is in infrared. You need to know how many frames of each class come from each lighting condition. Converting everything to grayscale (as you did) is the right call: it makes day and night frames look alike to the model. |
| **The civet crop is added** | In a wide room view the civet may cover only 2–5 % of the image. A 224×224 crop centred on it gives the classifier far more useful pixels. |
| **One CLAHE object is created once** | Faster than creating it for every frame. |

Why grayscale but **3 channels**: pretrained models (ResNet, EfficientNet, YOLO) expect 3-channel input, so the grey image is copied into all three.

In [ ]:
CLAHE = cv2.createCLAHE(clipLimit=1.2, tileGridSize=(8, 8))

def mask_text(frame, boxes, method=TEXT_MASK_METHOD):
    # Hide burned-in timestamp / camera name
    h, w = frame.shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    for x1, y1, x2, y2 in boxes:
        mask[int(y1 * h):int(y2 * h), int(x1 * w):int(x2 * w)] = 255
    if method == "inpaint":
        return cv2.inpaint(frame, mask, inpaintRadius=5, flags=cv2.INPAINT_TELEA)
    out = frame.copy()
    out[mask > 0] = 114
    return out

def enhance(frame):
    # Grayscale → median filter (removes IR speckle noise) → CLAHE (local contrast) → 3 channels
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 3)
    gray = CLAHE.apply(gray)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

def frame_quality(frame):
    # Measured on the masked RAW frame (before enhance)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    small = cv2.resize(frame, (320, int(320 * frame.shape[0] / frame.shape[1])), interpolation=cv2.INTER_AREA)
    b, g, r = cv2.split(small.astype(np.int16))
    colorfulness = float((np.abs(b - g) + np.abs(g - r)).mean() / 2)
    return {
        "brightness": round(float(gray.mean()), 2),
        "contrast": round(float(gray.std()), 2),
        "blur_score": round(float(cv2.Laplacian(gray, cv2.CV_64F).var()), 2),
        "colorfulness": round(colorfulness, 2),
        "lighting": "ir_night" if colorfulness < 3 else "color_day",
    }

def small_gray(frame):
    # Small grayscale version (320 px wide) used to compare consecutive frames
    h, w = frame.shape[:2]
    return cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (320, int(320 * h / w)), interpolation=cv2.INTER_AREA)

def motion_percent(current_small, previous_small, pixel_change=25):
    # % of pixels that changed clearly between two frames. A small moving civet changes few pixels,
    # so this is far more sensitive than the average difference over the whole room.
    diff = np.abs(current_small.astype(np.int16) - previous_small.astype(np.int16))
    return float((diff > pixel_change).mean() * 100)

def letterbox(image, target_size=FULL_SIZE, pad_value=114):
    # Resize keeping the aspect ratio, then pad with grey to a square (your original function)
    h, w = image.shape[:2]
    scale = target_size / max(h, w)
    new_w, new_h = round(w * scale), round(h * scale)
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR)
    left, top = (target_size - new_w) // 2, (target_size - new_h) // 2
    return cv2.copyMakeBorder(resized, top, target_size - new_h - top, left, target_size - new_w - left,
                              cv2.BORDER_CONSTANT, value=(pad_value,) * 3)

def crop_around_box(image, box, margin=CROP_MARGIN, size=CROP_SIZE, pad_value=114):
    # Square crop centred on the civet box, with margin; pads with grey if it runs off the image
    h, w = image.shape[:2]
    x1, y1, x2, y2 = box
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    side = max(x2 - x1, y2 - y1) * (1 + 2 * margin)
    side = int(max(side, 32))
    left, top = int(round(cx - side / 2)), int(round(cy - side / 2))
    pad = max(0, -left, -top, left + side - w, top + side - h)
    if pad:
        image = cv2.copyMakeBorder(image, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(pad_value,) * 3)
        left, top = left + pad, top + pad
    crop = image[top:top + side, left:left + side]
    interp = cv2.INTER_AREA if side > size else cv2.INTER_LINEAR
    return cv2.resize(crop, (size, size), interpolation=interp)

In [ ]:
# Preview every stage on 4 frames from different videos
sample = frames_df.groupby("video_id").head(1).head(4)
fig, axes = plt.subplots(len(sample), 3, figsize=(15, 3.6 * len(sample)))
axes = np.atleast_2d(axes)
for i, (_, r) in enumerate(sample.iterrows()):
    raw = cv2.imread(str(BASE_DIR / r.raw_path))
    masked = mask_text(raw, text_boxes_for(r.camera))
    stages = [("raw", raw), ("text masked", masked), ("enhanced + letterbox", letterbox(enhance(masked)))]
    q = frame_quality(masked)
    for j, (title, img) in enumerate(stages):
        axes[i, j].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[i, j].set_title(f"{r.video_id} · {title}" + (f"\n{q['lighting']}, blur={q['blur_score']:.0f}" if j == 1 else ""), fontsize=9)
        axes[i, j].axis("off")
plt.tight_layout()
plt.show()

## 7. Find the civet with MegaDetector

**MegaDetector** (Microsoft, free) is trained on millions of camera-trap images, including night-time infrared ones. It draws a box around anything that is an `animal`, `person` or `vehicle`. It does not know what a civet is, but it doesn't need to: there is only one animal in the enclosure.

What it gives you:
1. **The civet's box**, used to make the 224×224 crop. The classifier then looks at the civet, not at the room. This matters because if all "Eating" clips come from the feeding room, a full-frame model can learn *"feeding room = eating"* without looking at the animal.
2. **Frames where the civet isn't found.** These are copied to `review/no_civet_detected/` for you to check by eye. Don't treat them as "empty" automatically: in a Resting clip, a miss usually means the civet is curled up in a dark corner.
3. **Frames with a person** (keepers entering to feed or clean) are flagged so you can exclude them.

The first run downloads the model weights (a few hundred MB). If Colab asks you to restart the runtime after installing, restart and run all cells again.

In [ ]:
detector = None
if USE_MEGADETECTOR:
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "PytorchWildlife"], check=True)
    import torch
    from PytorchWildlife.models import detection as pw_detection
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    detector = pw_detection.MegaDetectorV6(device=DEVICE, pretrained=True, version=MD_VERSION)
    print("MegaDetector loaded on", DEVICE)
else:
    print("MegaDetector is off: no crops, no civet-present check.")

In [ ]:
ANIMAL, PERSON = 0, 1   # MegaDetector class ids

def detect_civet(frame):
    # Returns (best animal box [x1,y1,x2,y2] in pixels or None, its confidence, person_in_frame)
    if detector is None:
        return None, float("nan"), False
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = detector.single_image_detection(rgb, det_conf_thres=MD_CONF)
    det = result["detections"]
    person = bool((det.class_id == PERSON).any())
    is_animal = det.class_id == ANIMAL
    if not is_animal.any():
        return None, 0.0, person
    best = int(np.argmax(np.where(is_animal, det.confidence, -1)))   # one civet → keep the most confident box
    return [float(v) for v in det.xyxy[best]], float(det.confidence[best]), person

In [ ]:
# Check the detector on 8 random frames before running it on everything
if detector is not None:
    check = frames_df.sample(min(8, len(frames_df)), random_state=RANDOM_SEED)
    fig, axes = plt.subplots(2, 4, figsize=(18, 6))
    for ax in axes.reshape(-1):
        ax.axis("off")
    for ax, (_, r) in zip(axes.reshape(-1), check.iterrows()):
        masked = mask_text(cv2.imread(str(BASE_DIR / r.raw_path)), text_boxes_for(r.camera))
        box, conf, person = detect_civet(masked)
        shown = masked.copy()
        if box:
            cv2.rectangle(shown, tuple(map(int, box[:2])), tuple(map(int, box[2:])), (0, 255, 0), 3)
        ax.imshow(cv2.cvtColor(shown, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{r.label} · conf={conf:.2f}" + (" · PERSON" if person else ""), fontsize=9)
    plt.tight_layout()
    plt.show()

## 8. Run the full pipeline on every frame

For each frame: **read raw → mask text → measure quality → find civet → enhance → save full 640 → save crop 224 → write one manifest row.**

Nothing is deleted. Bad frames get **flags** in the manifest, and the model team filters with the `usable` column. That way, if a threshold turns out to be too strict, you change one number instead of re-running everything.

| Flag | Meaning |
|---|---|
| `too_dark` / `too_bright` / `low_contrast` | brightness < 20, > 235, or contrast < 5 (thresholds from your first notebook) |
| `blurry` | sharpness below half of that video's median. The threshold is relative because IR night frames are naturally softer than day frames. |
| `near_duplicate` | under 0.02 % of pixels changed since the previous frame (`motion_pct`). This is **normal for Resting**, so don't delete these. Only thin them out if a frame-by-frame classifier sees too many identical images. |
| `civet_detected` / `person_in_frame` | from MegaDetector |

In [ ]:
FULL_DIR = PROCESSED_DIR / f"full{FULL_SIZE}"
CROP_DIR = PROCESSED_DIR / f"crop{CROP_SIZE}"
NO_CIVET_DIR = REVIEW_DIR / "no_civet_detected"

rows = []
previous = {}   # video_id -> small gray image of the previous frame

for _, r in tqdm(list(frames_df.sort_values(["video_id", "timestamp_sec"]).iterrows()), desc="Frames"):
    raw = cv2.imread(str(BASE_DIR / r.raw_path))
    if raw is None:
        rows.append({**r.to_dict(), "readable": False})
        continue

    masked = mask_text(raw, text_boxes_for(r.camera))
    quality = frame_quality(masked)

    sg = small_gray(masked)
    motion = motion_percent(sg, previous[r.video_id]) if r.video_id in previous else np.nan
    previous[r.video_id] = sg

    box, conf, person = detect_civet(masked)
    enhanced = enhance(masked)

    full_path = FULL_DIR / r.label / r.raw_file
    full_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(full_path), letterbox(enhanced), [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])

    crop_rel = ""
    if box is not None:
        crop_path = CROP_DIR / r.label / r.raw_file
        crop_path.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(crop_path), crop_around_box(enhanced, box), [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
        crop_rel = str(crop_path.relative_to(BASE_DIR))
    elif detector is not None:
        review_path = NO_CIVET_DIR / r.label / r.raw_file
        review_path.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(review_path), masked, [cv2.IMWRITE_JPEG_QUALITY, 85])

    rows.append({
        **r.to_dict(), "readable": True, **quality,
        "motion_pct": round(motion, 4) if not np.isnan(motion) else np.nan,
        "civet_detected": (box is not None) if detector is not None else np.nan,
        "md_conf": round(conf, 4) if not np.isnan(conf) else np.nan,
        "box_x1": box[0] if box else np.nan, "box_y1": box[1] if box else np.nan,
        "box_x2": box[2] if box else np.nan, "box_y2": box[3] if box else np.nan,
        "person_in_frame": person,
        "full_path": str(full_path.relative_to(BASE_DIR)),
        "crop_path": crop_rel,
    })

manifest = pd.DataFrame(rows)
print(f"Processed {len(manifest):,} frames")

In [ ]:
m = manifest
m["too_dark"] = m.brightness < 20
m["too_bright"] = m.brightness > 235
m["low_contrast"] = m.contrast < 5
m["blurry"] = m.blur_score < 0.5 * m.groupby("video_id").blur_score.transform("median")
m["near_duplicate"] = m.motion_pct < MOTION_THRESHOLD

bad_image = m.too_dark | m.too_bright | m.low_contrast | m.blurry | ~m.readable.astype(bool)
if USE_MEGADETECTOR:
    m["usable"] = ~bad_image & (m.civet_detected == True) & ~m.person_in_frame.astype(bool)
else:
    m["usable"] = ~bad_image

flag_cols = ["too_dark", "too_bright", "low_contrast", "blurry", "near_duplicate", "person_in_frame", "usable"]
print("Share of frames with each flag, per label (%):")
print((m.groupby("label")[flag_cols].mean() * 100).round(1).to_string())
if USE_MEGADETECTOR:
    print("\nCivet detection rate per video (check any video below ~70 %):")
    print((m.groupby(["label", "video_id"]).civet_detected.mean() * 100).round(0).to_string())

## 9. Split by video — the most important ML decision in your part

If frames are shuffled randomly into train/test, frame 120 of video L03 is in training and frame 121 (almost the same image) is in test. The model "recognises" it, test accuracy looks like 98 %, and then it fails on new CCTV footage. **Whole videos go into one split.**

With 22 videos the split is small (about 3/1/1 for Eating), so a single test set gives a noisy score. The `fold` column also supports **5-fold cross-validation by video**: train 5 times, each time holding out one fold, and report the average.

The split is saved to `metadata/video_splits.csv` and **reused on every re-run**, so the test set never changes. Nobody should tune anything on test-set results.

In [ ]:
splits_csv = METADATA_DIR / "video_splits.csv"
processed_videos = m[["video_id", "label"]].drop_duplicates()

if splits_csv.exists():
    splits = pd.read_csv(splits_csv, dtype={"video_id": str})
    new_videos = processed_videos[~processed_videos.video_id.isin(splits.video_id)]
    if len(new_videos):
        print("New videos not in the saved split (assigned to train; move them by hand if needed):",
              new_videos.video_id.tolist())
        splits = pd.concat([splits, new_videos.assign(split="train", fold=-1)], ignore_index=True)
else:
    rng = np.random.default_rng(RANDOM_SEED)
    split_rows = []
    for label, group in processed_videos.groupby("label"):
        ids = sorted(group.video_id)
        rng.shuffle(ids)
        n = len(ids)
        n_test = max(1, round(0.2 * n)) if n >= 3 else 0
        n_val = max(1, round(0.15 * n)) if n >= 3 else 0
        for i, vid in enumerate(ids):
            split = "test" if i < n_test else ("val" if i < n_test + n_val else "train")
            split_rows.append({"video_id": vid, "label": label, "split": split, "fold": i % N_FOLDS})
    splits = pd.DataFrame(split_rows)
splits.to_csv(splits_csv, index=False)

m = m.drop(columns=["split", "fold"], errors="ignore").merge(splits[["video_id", "split", "fold"]], on="video_id", how="left")
m.to_csv(METADATA_DIR / "manifest.csv", index=False)
manifest = m

print("Videos per label and split:")
print(pd.crosstab(splits.label, splits.split).to_string(), "\n")
print("Usable frames per label and split:")
print(pd.crosstab(m[m.usable].label, m[m.usable].split).to_string())

## 10. Sanity checks before you hand anything over

Each table answers a question your supervisor will ask:

1. **Class balance**: are frame counts very uneven? Then the model team needs class weights or a frame cap.
2. **Class × camera** *(the shortcut check)*: if one class only ever comes from one camera, the model may learn the room instead of the behaviour. Fix it with crops (done), and ideally collect e.g. Resting/Locomotion clips **from the feeding room** too.
3. **Class × lighting**: if all Eating frames are daytime and all Resting frames are night, the model can learn "dark = resting". Grayscale + CLAHE reduces this but does not remove it.

In [ ]:
usable = m[m.usable]
print("1) Usable frames per label:")
print(usable.label.value_counts().to_string(), "\n")

print("2) Videos per label × camera:")
print(pd.crosstab(m.drop_duplicates("video_id").label, m.drop_duplicates("video_id").camera.replace("", "unknown")).to_string(), "\n")

print("3) Usable frames per label × lighting:")
print(pd.crosstab(usable.label, usable.lighting).to_string())

In [ ]:
# What the model will actually see: random crops (or full frames) per label
column = "crop_path" if USE_MEGADETECTOR else "full_path"
for label, group in usable.groupby("label"):
    has_image = group[group[column] != ""]
    picks = has_image.sample(min(12, len(has_image)), random_state=RANDOM_SEED)
    print(label)
    show_grid([BASE_DIR / p for p in picks[column]], [f"{v} t={t:.0f}s" for v, t in zip(picks.video_id, picks.timestamp_sec)])

## 11. Hand-off to the model team

**`metadata/manifest.csv` — the important columns**

| Column | Use |
|---|---|
| `video_id`, `label`, `camera` | identity; **always group by `video_id`** when splitting |
| `timestamp_sec`, `frame_idx`, `window_id` | order frames and build 1-second sequences for CNN+LSTM |
| `full_path` / `crop_path` | the image to train on (paths are relative to `MyDrive/CMZ/`) |
| `usable` + the individual flags | filter bad frames |
| `split`, `fold` | train/val/test and cross-validation folds (by video) |
| `box_x1…box_y2`, `md_conf` | civet location in the raw frame |

**What happens in the training code, not here**
* **Normalisation** (ImageNet mean/std) is done in the data loader. Saved images stay as normal 0–255 JPEGs.
* **Augmentation** only on the *train* split, applied randomly each epoch: horizontal flip, brightness/contrast jitter, small rotation/zoom, a little noise. No vertical flips (civets don't walk on the ceiling).
* **Inference must reuse** `mask_text → enhance → detect_civet → crop_around_box` (or `letterbox`). Copy these functions into the deployment code unchanged.

The cell below shows the model team how to turn the manifest into 1-second sequences.

In [ ]:
# Example: build 1-second sequences (windows) for a CNN+LSTM model
min_frames = int(SAMPLE_FPS * WINDOW_SEC * 0.8)   # allow one missing frame per window
image_col = "crop_path" if USE_MEGADETECTOR else "full_path"

windows = (
    usable.sort_values(["video_id", "timestamp_sec"])
          .groupby(["video_id", "window_id"])
          .agg(label=("label", "first"), split=("split", "first"), fold=("fold", "first"),
               n_frames=(image_col, "count"), frames=(image_col, list))
          .reset_index()
)
windows = windows[windows.n_frames >= min_frames]
windows.to_csv(METADATA_DIR / "windows_1s.csv", index=False)

print(f"{len(windows):,} usable 1-second windows")
print(pd.crosstab(windows.label, windows.split).to_string())

In [ ]:
# Optional: zip processed images + metadata into one file to share with the team (raw videos not included)
import zipfile
archive = BASE_DIR / "cmz_processed_dataset.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_STORED) as zf:   # JPEGs are already compressed
    for folder in [PROCESSED_DIR, METADATA_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                zf.write(path, path.relative_to(BASE_DIR))
print(f"Saved {archive}  ({archive.stat().st_size / 1e6:.1f} MB)")